In [1]:
import os
import pandas as pd
import numpy as np
os.chdir("../website")
from models import Database
from dotenv import load_dotenv
load_dotenv()

db = Database(
    host=os.environ["HOST"],
    port=os.environ["PORT"],
    database=os.environ["DATABASE"],
    user=os.environ["USER"],
    password=os.environ["PASSWORD"]
    )

In [2]:
filename = 'stock 1 Nov 2025'
input_file = rf'C:\Users\keong\Downloads\{filename}.xlsx'
output_file = rf'C:\Users\keong\Downloads\{filename}_no_returned.xlsx'
relabeled_file = rf'C:\Users\keong\Downloads\{filename}_relabeled.xlsx'

In [10]:
df = pd.read_excel(input_file)
df = df.loc[df['stk_returned']!=1]
df['stk_type'] = df['stk_type'].str.upper()
df['pur_code'] = df['pur_code'].str.upper()
distinct_pur = db.select(table='purchase',columns=['pur_id','pur_code','pur_gold_cost','pur_gold_cost_999','pur_date'])
merged = pd.merge(df,distinct_pur,on='pur_code',how='left')
## stk_gold_cost conditions
conditions = [
    merged['stk_gold_type'] == 916,
    merged['stk_gold_type'] == 999
]
choices = [
    merged['pur_gold_cost'],
    merged['pur_gold_cost_999']
]

merged['stk_pur_id'] = merged['pur_id']
merged['stk_gold_cost'] = np.select(conditions, choices, default=None)
merged['stk_pur_date'] = merged['pur_date']
merged['stk_status'] = 'IN STOCK'
merged.drop(columns=['pur_id','pur_gold_cost','pur_code','pur_gold_cost_999','pur_date'],inplace=True)
# merged.drop(columns=['pur_id','pur_gold_cost','pur_gold_cost_999','pur_date'],inplace=True)
merged.to_excel(output_file,index=False)

## Generate Barcode File

In [25]:
stk_id = tuple(db.select(table='stock',columns=['stk_id'],where="date(stk_created_at) = '2025-10-27'")['stk_id'].unique())

In [31]:
query = """
        SELECT 
            stk.stk_id,
            stk.stk_barcode,
            stk.stk_barcode as stk_barcode_text,
            stk.stk_weight,
            COALESCE(stk.stk_length, stk.stk_size) AS stk_length_size,
            stk.stk_returned,
            slm.slm_name,
            '''' || TO_CHAR(p.pur_date, 'MMYY') AS stk_pur_monthyear
        FROM konghin.stock stk
        LEFT JOIN konghin.purchase p ON stk.stk_pur_id = p.pur_id
        LEFT JOIN konghin.salesman slm ON p.pur_slm_id = slm.slm_id
        WHERE stk.stk_id IN {stk_id}
    """.format(stk_id = str(stk_id))

In [38]:
result = db.select_raw(query)
result

,stk_id,stk_barcode,stk_barcode_text,stk_weight,stk_length_size,stk_returned,slm_name,stk_pur_monthyear
0,STK_100740,JB8600120740,JB8600120740,1.27,9,0,YM,'1223
1,STK_100741,JB8600120741,JB8600120741,1.51,15,0,YM,'1223
2,STK_100742,JB8600120742,JB8600120742,1.44,14,0,YM,'1223
3,STK_100743,JD6200160743,JD6200160743,2.03,15,0,MKT,'0925
4,STK_100744,JD6200160744,JD6200160744,1.94,10,0,MKT,'0925
...,...,...,...,...,...,...,...,...
133,STK_100873,JB3200130873,JB3200130873,0.93,None,0,YX,'0122
134,STK_100874,JB3200150874,JB3200150874,1.08,None,0,YX,'0122
135,STK_100875,JB3200130875,JB3200130875,1.55,None,0,YX,'0122
136,STK_100883,JC3100410883,JC3100410883,1.63,None,0,YM,'0524


In [43]:
df = result.copy()

# Step 1: Split the DataFrame into pairs of rows
df_even = df.iloc[::2].reset_index(drop=True)  # 0, 2, 4, ...
df_odd = df.iloc[1::2].reset_index(drop=True)  # 1, 3, 5, ...

# Step 2: Rename odd columns with suffix "_2"
df_odd = df_odd.add_suffix("_2")

# Step 3: Concatenate side by side
df_combined = pd.concat([df_even, df_odd], axis=1)

# ✅ Final result
display(df_combined)

df_combined.to_excel(relabeled_file,index=False)


,stk_id,stk_barcode,stk_barcode_text,stk_weight,stk_length_size,stk_returned,slm_name,stk_pur_monthyear,stk_id_2,stk_barcode_2,stk_barcode_text_2,stk_weight_2,stk_length_size_2,stk_returned_2,slm_name_2,stk_pur_monthyear_2
0,STK_100740,JB8600120740,JB8600120740,1.27,9,0,YM,'1223,STK_100741,JB8600120741,JB8600120741,1.51,15,0,YM,'1223
1,STK_100742,JB8600120742,JB8600120742,1.44,14,0,YM,'1223,STK_100743,JD6200160743,JD6200160743,2.03,15,0,MKT,'0925
2,STK_100744,JD6200160744,JD6200160744,1.94,10,0,MKT,'0925,STK_100745,JD6200160745,JD6200160745,1.88,13,0,MKT,'0925
3,STK_100746,JD6200160746,JD6200160746,1.89,8,0,MKT,'0925,STK_100747,JD2200160747,JD2200160747,2.05,14,0,MKT,'0525
4,STK_100748,JD2200160748,JD2200160748,2.02,12,0,MKT,'0525,STK_100749,JD2200160749,JD2200160749,1.95,9,0,MKT,'0525
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
64,STK_100868,JA6100100868,JA6100100868,1.18,7,0,XW,'0119,STK_100869,JB3200130869,JB3200130869,1.1,None,0,YX,'0122
65,STK_100870,JB3200130870,JB3200130870,1.28,None,0,YX,'0122,STK_100871,JB3200130871,JB3200130871,1.11,None,0,YX,'0122
66,STK_100872,JB3200130872,JB3200130872,1.26,None,0,YX,'0122,STK_100873,JB3200130873,JB3200130873,0.93,None,0,YX,'0122
67,STK_100874,JB3200150874,JB3200150874,1.08,None,0,YX,'0122,STK_100875,JB3200130875,JB3200130875,1.55,None,0,YX,'0122
